# LeetCode 757 — Set Intersection Size At Least Two bằng Python

- Link: https://leetcode.com/problems/set-intersection-size-at-least-two/
- Độ khó: **Hard**
- Chủ đề: Tham lam (Greedy) trên đoạn, Sắp xếp, Bài toán "điểm phủ đoạn" (hitting set), Ràng buộc hiệu (difference constraints)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def intersectionSizeTwo(self, intervals: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Cho nhiều đoạn số nguyên. Ta cần chọn **ít điểm nguyên nhất** sao cho **mỗi đoạn chứa ít nhất 2 điểm được chọn**. Đây là phiên bản "nâng cấp" của bài kinh điển *452. Minimum Number of Arrows to Burst Balloons* (mỗi đoạn chứa ít nhất **1** điểm).

Bài dạy một kỹ năng quan trọng: **thiết kế và chứng minh thuật toán tham lam** — sắp xếp theo tiêu chí nào, chọn điểm nào, và vì sao lựa chọn đó không bao giờ thua.

Sau notebook này bạn sẽ:

1. Viết lời giải vét cạn trên toạ độ nhỏ để có "chuẩn vàng" kiểm tra.
2. Hiểu tham lam "chọn điểm **càng về phải càng tốt**" cho bài 1 điểm, rồi mở rộng sang 2 điểm.
3. Nắm chi tiết tinh tế: **sắp theo đầu mút phải tăng dần, hoà thì đầu mút trái giảm dần**, và chỉ cần nhớ **2 điểm lớn nhất** đã chọn.
4. Biết lời giải tổng quát "mỗi đoạn cần `k` điểm" bằng mảng `need[]` trong `O(n^2)`.
5. Nhìn bài toán dưới góc **ràng buộc hiệu** (difference constraints) và giải bằng đường đi dài nhất (Bellman–Ford).

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `intervals`, mỗi phần tử `[start, end]` biểu diễn **mọi số nguyên** từ `start` đến `end` (tính cả hai đầu). Một tập `nums` gọi là **tập chứa** nếu mỗi đoạn có **ít nhất 2 phần tử** thuộc `nums`. Trả về **kích thước nhỏ nhất** của một tập chứa.

**Ví dụ 1**

```text
Input: intervals = [[1,3],[3,7],[8,9]]
Output: 5
Giải thích: nums = [2, 3, 4, 8, 9]. Đoạn [1,3] chứa 2,3; [3,7] chứa 3,4; [8,9] chứa 8,9.
Không thể dùng 4 điểm.
```

**Ví dụ 2**

```text
Input: intervals = [[1,3],[1,4],[2,5],[3,5]]
Output: 3
Giải thích: nums = [2, 3, 4].
```

**Ví dụ 3**

```text
Input: intervals = [[1,2],[2,3],[2,4],[4,5]]
Output: 5
Giải thích: nums = [1, 2, 3, 4, 5].
```

**Ràng buộc**

- `1 <= intervals.length <= 3000`
- `intervals[i].length == 2`
- `0 <= start_i < end_i <= 10^8`  (mỗi đoạn có ít nhất 2 số nguyên)

In [1]:
from typing import List

EXAMPLES = [
    ([[1, 3], [3, 7], [8, 9]], 5),
    ([[1, 3], [1, 4], [2, 5], [3, 5]], 3),
    ([[1, 2], [2, 3], [2, 4], [4, 5]], 5),
]

def check_cover(intervals, nums):
    """Mỗi đoạn có chứa >= 2 phần tử của nums không?"""
    S = set(nums)
    return all(sum(1 for x in S if s <= x <= e) >= 2 for s, e in intervals)

print(check_cover([[1, 3], [3, 7], [8, 9]], [2, 3, 4, 8, 9]))
print(check_cover([[1, 3], [1, 4], [2, 5], [3, 5]], [2, 3, 4]))
print(check_cover([[1, 3], [3, 7], [8, 9]], [2, 3, 8, 9]))   # [3,7] chỉ chứa 3

True
True
False


## 3. Kiến thức nền

### 3.1. Sắp xếp với khoá nhiều tiêu chí

`sorted(a, key=lambda x: (x[1], -x[0]))`: so sánh theo `x[1]` tăng dần trước; nếu bằng nhau thì theo `-x[0]` tăng dần, tức `x[0]` **giảm dần**. Tuple được so sánh theo thứ tự từ điển.

In [2]:
iv = [[1, 5], [3, 5], [2, 4], [0, 5]]
print(sorted(iv, key=lambda x: (x[1], -x[0])))
print(sorted(iv, key=lambda x: x[1]))          # hoà giữ nguyên thứ tự cũ (stable)

[[2, 4], [3, 5], [1, 5], [0, 5]]
[[2, 4], [1, 5], [3, 5], [0, 5]]


### 3.2. Kiểm tra điểm thuộc đoạn

Điểm `x` thuộc đoạn `[s, e]` khi `s <= x <= e`. Python cho phép viết chuỗi so sánh `s <= x <= e` rất gọn.

In [3]:
s, e = 3, 7
print([x for x in range(0, 10) if s <= x <= e])

[3, 4, 5, 6, 7]


### 3.3. Duyệt tập con theo kích thước tăng dần (`itertools.combinations`)

Để vét cạn "tập nhỏ nhất thoả điều kiện", ta thử mọi tập con kích thước 1, 2, 3, ... — tập đầu tiên thoả mãn cho đáp án. `combinations(range(L), r)` sinh mọi tập con kích thước `r`.

In [4]:
from itertools import combinations
print(list(combinations([1, 2, 3, 4], 2)))
print(sum(1 for _ in combinations(range(10), 5)))   # C(10,5) = 252

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
252


### 3.4. Bài nền: mỗi đoạn cần **1** điểm (LeetCode 452)

Tham lam: sắp theo `end` tăng; giữ `last` = điểm vừa chọn. Đoạn nào chưa chứa `last` (`s > last`) thì chọn điểm **`end`** của nó (điểm xa nhất về phải trong đoạn — có cơ hội phủ nhiều đoạn sau nhất).

In [5]:
def min_points_one(intervals):
    last, cnt = float('-inf'), 0
    for s, e in sorted(intervals, key=lambda x: x[1]):
        if s > last:
            last = e
            cnt += 1
    return cnt

print(min_points_one([[10, 16], [2, 8], [1, 6], [7, 12]]))   # 2 (chọn 6 và 12)

2


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Tính tay ví dụ 1: `[[1,3],[3,7],[8,9]]`

- Đoạn `[1,3]` cần 2 điểm. Chọn điểm nào? Đoạn sau `[3,7]` bắt đầu ở 3, nên chọn **càng về phải càng tốt**: `2, 3`.
- `[3,7]`: đã chứa `3` (1 điểm). Cần thêm 1 điểm → chọn `7` (xa nhất). Tập: `{2, 3, 7}`.
- `[8,9]`: chưa chứa điểm nào → chọn `8, 9`. Tổng 5.

(Đề ghi `[2,3,4,8,9]`, ta ra `{2,3,7,8,9}` — cũng 5 điểm; đáp án chỉ hỏi **kích thước**.)

### 4.2. Tính tay ví dụ 2: `[[1,3],[1,4],[2,5],[3,5]]`

Sắp theo `end`: `[1,3]`, `[1,4]`, rồi hai đoạn kết thúc tại 5. Chọn `2, 3` cho `[1,3]`. `[1,4]` đã có `2, 3`. Đoạn kết thúc ở 5: `[3,5]` có `3` → thêm `5`; `[2,5]` có `3, 5`. Tổng 3.

### 4.3. Quan sát then chốt

Khi xử lý các đoạn theo `end` tăng dần, mọi điểm đã chọn đều `<= end` hiện tại. Với đoạn `[s, e]`, số điểm đã chọn nằm trong nó chỉ phụ thuộc vào **những điểm lớn nhất** đã chọn. Vì chỉ cần 2 điểm, ta chỉ cần nhớ **2 điểm lớn nhất** `a < b`.

In [6]:
nums = {2, 3, 7, 8, 9}
print(check_cover([[1, 3], [3, 7], [8, 9]], nums), len(nums))

True 5


## 5. Cách 1 — Brute Force: thử mọi tập con theo kích thước

**Ý tưởng.** Một tập chứa tối ưu luôn có thể chọn điểm trong `[min start, max end]`. Thử mọi tập con các điểm trong đoạn đó với kích thước `r = 1, 2, ...`; tập đầu tiên phủ được mọi đoạn cho đáp án. Chỉ dùng được khi toạ độ rất nhỏ (≤ ~14 điểm).

In [7]:
def brute(intervals):
    lo = min(s for s, _ in intervals)
    hi = max(e for _, e in intervals)
    pts = list(range(lo, hi + 1))
    for r in range(len(pts) + 1):
        for comb in combinations(pts, r):
            if all(sum(1 for x in comb if s <= x <= e) >= 2 for s, e in intervals):
                return r
    return -1   # không xảy ra vì mỗi đoạn có >= 2 số nguyên

for iv, exp in EXAMPLES:
    print(iv, "->", brute(iv), "(kỳ vọng", exp, ")")

[[1, 3], [3, 7], [8, 9]] -> 5 (kỳ vọng 5 )
[[1, 3], [1, 4], [2, 5], [3, 5]] -> 3 (kỳ vọng 3 )
[[1, 2], [2, 3], [2, 4], [4, 5]] -> 5 (kỳ vọng 5 )


**Độ phức tạp.** Với `L = hi - lo + 1` điểm: tối đa `2^L` tập con, mỗi tập kiểm tra `O(n · L)` → `O(2^L · n · L)`. Toạ độ tới `10^8` → hoàn toàn bất khả thi.

**Nút thắt.** Ta thử mù quáng mọi tổ hợp. Cần một quy tắc **chọn điểm có định hướng**: chọn điểm sao cho có ích nhất cho các đoạn còn lại.

## 6. Bước đột phá

**Insight 1 — Xử lý theo đầu mút phải tăng dần.** Đoạn có `end` nhỏ nhất là đoạn "gấp" nhất: mọi điểm phủ nó đều `<= end`. Trong các điểm có thể dùng cho nó, điểm **càng lớn** càng có khả năng nằm trong các đoạn sau (vì các đoạn sau có `end` lớn hơn). Vậy nếu cần thêm điểm, chọn `e`, rồi `e - 1`.

**Insight 2 — Chỉ nhớ hai điểm lớn nhất `a < b`.** Mọi điểm đã chọn `<= e` hiện tại, và các điểm được thêm luôn lớn dần. Đoạn `[s, e]`:

- Nếu `s <= a`: chứa cả `a` và `b` (vì `a < b <= e`) → **đủ**, bỏ qua.
- Nếu `a < s <= b`: chỉ chứa `b` → thêm **1** điểm `e`; cập nhật `a, b = b, e`.
- Nếu `s > b`: không chứa điểm nào → thêm **2** điểm `e - 1, e`; cập nhật `a, b = e - 1, e`.

**Insight 3 — Hoà `end` thì `start` giảm dần.** Hai đoạn cùng `end`: xử lý đoạn **ngắn hơn** (start lớn hơn) trước. Đoạn ngắn khó hơn; điểm chọn cho nó (`e`, `e-1`) chắc chắn cũng nằm trong đoạn dài hơn. Nếu làm ngược, có thể thêm `e` cho đoạn dài rồi lại cần thêm `e` lần nữa cho đoạn ngắn (trùng điểm!).

**Bất biến.** Sau khi xử lý các đoạn đầu tiên theo thứ tự đã sắp: (1) mọi đoạn đã xử lý chứa ≥ 2 điểm đã chọn; (2) `a < b` là hai điểm lớn nhất đã chọn; (3) số điểm đã chọn là nhỏ nhất có thể cho các đoạn đó, và trong mọi lời giải tối ưu, hai điểm lớn nhất không thể "lớn hơn" `(a, b)` (tập của ta "đẩy về phải" tối đa).

**Quy tắc nếu… thì…**

- Nếu đoạn đã chứa cả hai điểm lớn nhất → bỏ qua.
- Nếu chỉ chứa điểm lớn nhất → thêm `end`.
- Nếu không chứa điểm nào → thêm `end - 1` và `end`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam với hai điểm lớn nhất `(a, b)`

**Giải thích.** Sắp `intervals` theo `(end, -start)`. Khởi tạo `a = b = -1` (nhỏ hơn mọi toạ độ vì `start >= 0`). Duyệt và áp dụng 3 trường hợp ở mục 6.

In [8]:
def greedy_two(intervals):
    a, b = -1, -1          # hai điểm lớn nhất đã chọn, a < b
    ans = 0
    for s, e in sorted(intervals, key=lambda x: (x[1], -x[0])):
        if s <= a:
            continue                  # đã có 2 điểm
        if s <= b:
            ans += 1                  # có 1 điểm (b), thêm e
            a, b = b, e
        else:
            ans += 2                  # chưa có điểm nào, thêm e-1, e
            a, b = e - 1, e
    return ans

for iv, exp in EXAMPLES:
    print(greedy_two(iv), exp)

5 5
3 3
5 5


**Code nộp LeetCode (Cách 2).**

In [9]:
class SolutionGreedy:
    def intersectionSizeTwo(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: (x[1], -x[0]))
        a = b = -1
        ans = 0
        for s, e in intervals:
            if s <= a:
                continue
            if s <= b:
                ans += 1
                a, b = b, e
            else:
                ans += 2
                a, b = e - 1, e
        return ans

print([SolutionGreedy().intersectionSizeTwo([r[:] for r in iv]) for iv, _ in EXAMPLES])

[5, 3, 5]


**Độ phức tạp.** Sắp xếp `O(n log n)`, duyệt `O(n)`. Bộ nhớ `O(1)` ngoài phần sắp xếp (Timsort cần `O(n)`).

### 7.2. Cách 3 — Tham lam tổng quát với mảng `need[]` (mỗi đoạn cần `k` điểm)

**Giải thích.** Cách này mở rộng tự nhiên sang "mỗi đoạn cần ít nhất `k` điểm" và **in ra luôn tập điểm**.

- Sắp theo `(end, -start)`. `need[i] = k` cho mọi đoạn.
- Duyệt đoạn `i`. Nếu `need[i] > 0`: chọn `need[i]` điểm **lớn nhất chưa chọn** trong `[s_i, e_i]`, bắt đầu từ `e_i` đi xuống. Với mỗi điểm `p` mới chọn, giảm `need[j]` của mọi đoạn `j` (chưa xử lý) chứa `p`.

Vì các đoạn sau có `end >= e_i`, một điểm `p <= e_i` nằm trong đoạn `j` khi và chỉ khi `s_j <= p`.

Để tìm "điểm lớn nhất chưa chọn", ta dùng `set` các điểm đã chọn và đi xuống từ `e_i`. Vì `need[i] <= k` và các điểm đã chọn trong `[s_i, e_i]` ít, bước này nhanh.

In [10]:
def greedy_need(intervals, k=2, return_points=False):
    iv = sorted(intervals, key=lambda x: (x[1], -x[0]))
    n = len(iv)
    need = [k] * n
    chosen = set()
    for i, (s, e) in enumerate(iv):
        p = e
        while need[i] > 0:
            while p in chosen:
                p -= 1
            # p chưa chọn và p >= s (vì đoạn có đủ chỗ: e - s + 1 >= k)
            chosen.add(p)
            for j in range(i, n):           # cập nhật mọi đoạn từ i trở đi chứa p
                if iv[j][0] <= p <= iv[j][1]:
                    need[j] -= 1
    return (len(chosen), sorted(chosen)) if return_points else len(chosen)

for iv, exp in EXAMPLES:
    print(greedy_need(iv, return_points=True), exp)

(5, [2, 3, 7, 8, 9]) 5
(3, [2, 3, 5]) 3
(5, [1, 2, 3, 4, 5]) 5


> Lưu ý: với `k > 2`, đề phải đảm bảo mỗi đoạn có ít nhất `k` số nguyên. Ở bài gốc `k = 2` và `start < end` nên luôn đúng.

**Code nộp LeetCode (Cách 3).**

In [11]:
class SolutionNeed:
    def intersectionSizeTwo(self, intervals: List[List[int]]) -> int:
        intervals.sort(key=lambda x: (x[1], -x[0]))
        n = len(intervals)
        need = [2] * n
        ans = 0
        chosen = set()
        for i in range(n):
            s, e = intervals[i]
            p = e
            while need[i] > 0:
                while p in chosen:
                    p -= 1
                chosen.add(p)
                ans += 1
                for j in range(i, n):
                    if intervals[j][0] <= p:     # end_j >= e >= p
                        need[j] -= 1
        return ans

print([SolutionNeed().intersectionSizeTwo([r[:] for r in iv]) for iv, _ in EXAMPLES])

[5, 3, 5]


**Độ phức tạp.** Mỗi điểm chọn quét `O(n)` đoạn, có tối đa `2n` điểm → `O(n^2)` = `9·10^6` thao tác với `n = 3000` — chấp nhận được. Bộ nhớ `O(n)`.

### 7.3. Cách 4 — Ràng buộc hiệu (difference constraints) + đường đi dài nhất

**Giải thích.** Gọi `P(x)` = số điểm được chọn **nhỏ hơn** `x` (hàm tổng tiền tố). Khi đó:

- Đoạn `[s, e]` chứa ≥ 2 điểm ⇔ `P(e + 1) - P(s) >= 2`.
- Mỗi số nguyên được chọn tối đa một lần ⇔ `0 <= P(x + 1) - P(x) <= 1`.
- Mục tiêu: **cực tiểu** `P(max_end + 1) - P(min_start)`.

Nén toạ độ về các mốc `c_0 < c_1 < ... < c_m` gồm mọi `s` và `e + 1`. Giữa hai mốc liền kề: `0 <= P(c_{t+1}) - P(c_t) <= c_{t+1} - c_t`.

Mỗi ràng buộc `P(v) - P(u) >= w` là một cạnh `u → v` trọng số `w`. Giá trị nhỏ nhất thoả mãn mọi ràng buộc (với `P(c_0) = 0`) chính là **độ dài đường đi dài nhất** từ `c_0` — tính bằng Bellman–Ford (relax tới khi ổn định). Cạnh ngược `c_{t+1} → c_t` trọng số `-(gap)` mã hoá ràng buộc `<= gap`.

Cách này **chậm** (`O(V · E)` trong trường hợp xấu) nhưng là một góc nhìn mạnh: đổi bài tối ưu tổ hợp thành bài đồ thị.

In [12]:
def diff_constraints(intervals):
    keys = sorted({s for s, _ in intervals} | {e + 1 for _, e in intervals})
    idx = {x: i for i, x in enumerate(keys)}
    m = len(keys)
    edges = []
    for s, e in intervals:
        edges.append((idx[s], idx[e + 1], 2))            # P(e+1) - P(s) >= 2
    for t in range(m - 1):
        gap = keys[t + 1] - keys[t]
        edges.append((t, t + 1, 0))                       # P tăng không giảm
        edges.append((t + 1, t, -gap))                    # tăng tối đa gap
    dist = [float('-inf')] * m
    dist[0] = 0
    for _ in range(m):                                    # Bellman-Ford (đường dài nhất)
        changed = False
        for u, v, w in edges:
            if dist[u] != float('-inf') and dist[u] + w > dist[v]:
                dist[v] = dist[u] + w
                changed = True
        if not changed:
            break
    return dist[-1]

for iv, exp in EXAMPLES:
    print(diff_constraints(iv), exp)

5 5
3 3
5 5


**Code nộp LeetCode (Cách 4)** — đúng nhưng có thể **chậm** với `n = 3000` ở trường hợp xấu; để học ý tưởng. Ta sắp các cạnh theo thứ tự mốc để Bellman–Ford hội tụ nhanh hơn (giống SPFA theo thứ tự).

In [13]:
class SolutionDiff:
    def intersectionSizeTwo(self, intervals: List[List[int]]) -> int:
        keys = sorted({s for s, _ in intervals} | {e + 1 for _, e in intervals})
        idx = {x: i for i, x in enumerate(keys)}
        m = len(keys)
        fwd = [[] for _ in range(m)]          # cạnh đi tới mốc lớn hơn
        for s, e in intervals:
            fwd[idx[s]].append((idx[e + 1], 2))
        for t in range(m - 1):
            fwd[t].append((t + 1, 0))
        NEG = float('-inf')
        dist = [NEG] * m
        dist[0] = 0
        while True:
            changed = False
            for u in range(m):                       # quét xuôi: cạnh tiến
                if dist[u] == NEG:
                    continue
                for v, w in fwd[u]:
                    if dist[u] + w > dist[v]:
                        dist[v] = dist[u] + w
                        changed = True
            for t in range(m - 2, -1, -1):           # quét ngược: cạnh lùi -gap
                gap = keys[t + 1] - keys[t]
                if dist[t + 1] - gap > dist[t]:
                    dist[t] = dist[t + 1] - gap
                    changed = True
            if not changed:
                return dist[-1]

print([SolutionDiff().intersectionSizeTwo([r[:] for r in iv]) for iv, _ in EXAMPLES])

[5, 3, 5]


**Độ phức tạp.** Mỗi vòng quét `O(n)`; số vòng tối đa `O(n)` (thực tế ít) → `O(n^2)` xấu nhất cho bản quét xuôi/ngược. Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace **Cách 2** trên ví dụ 3: `[[1,2],[2,3],[2,4],[4,5]]`. Sắp theo `(end, -start)`: `[1,2], [2,3], [2,4], [4,5]`.

| đoạn | a | b | trường hợp | thêm | a', b' | ans |
|---|---|---|---|---|---|---|
| [1,2] | -1 | -1 | s=1 > b | 1, 2 | 1, 2 | 2 |
| [2,3] | 1 | 2 | a < s=2 <= b | 3 | 2, 3 | 3 |
| [2,4] | 2 | 3 | s=2 <= a | — | 2, 3 | 3 |
| [4,5] | 2 | 3 | s=4 > b | 4, 5 | 4, 5 | 5 |

Đáp án 5, tập `{1,2,3,4,5}`.

In [14]:
def trace_greedy_two(intervals):
    a, b, ans = -1, -1, 0
    order = sorted(intervals, key=lambda x: (x[1], -x[0]))
    print("thứ tự đã sắp:", order)
    print(f"{'đoạn':>8} | {'a':>3} {'b':>3} | {'trường hợp':<16} | thêm     | a' b' | ans")
    for s, e in order:
        if s <= a:
            case, add = "s <= a (đủ)", []
        elif s <= b:
            case, add = "a < s <= b", [e]
        else:
            case, add = "s > b", [e - 1, e]
        old = (a, b)
        if len(add) == 1:
            a, b = b, e
        elif len(add) == 2:
            a, b = e - 1, e
        ans += len(add)
        print(f"{str([s, e]):>8} | {old[0]:>3} {old[1]:>3} | {case:<16} | {str(add):<8} | {a:>2} {b:>2} | {ans}")
    return ans

trace_greedy_two([[1, 2], [2, 3], [2, 4], [4, 5]])

thứ tự đã sắp: [[1, 2], [2, 3], [2, 4], [4, 5]]
    đoạn |   a   b | trường hợp       | thêm     | a' b' | ans
  [1, 2] |  -1  -1 | s > b            | [1, 2]   |  1  2 | 2
  [2, 3] |   1   2 | a < s <= b       | [3]      |  2  3 | 3
  [2, 4] |   2   3 | s <= a (đủ)      | []       |  2  3 | 3
  [4, 5] |   2   3 | s > b            | [4, 5]   |  4  5 | 5


5

Trace ví dụ 2 — quan sát vai trò của **hoà `end`, `start` giảm dần**: `[3,5]` được xử lý trước `[2,5]`.

In [15]:
trace_greedy_two([[1, 3], [1, 4], [2, 5], [3, 5]])

thứ tự đã sắp: [[1, 3], [1, 4], [3, 5], [2, 5]]
    đoạn |   a   b | trường hợp       | thêm     | a' b' | ans
  [1, 3] |  -1  -1 | s > b            | [2, 3]   |  2  3 | 2
  [1, 4] |   2   3 | s <= a (đủ)      | []       |  2  3 | 2
  [3, 5] |   2   3 | a < s <= b       | [5]      |  3  5 | 3
  [2, 5] |   3   5 | s <= a (đủ)      | []       |  3  5 | 3


3

Trace **Cách 3** (mảng `need`) trên ví dụ 1, in `need` sau mỗi điểm được chọn.

In [16]:
def trace_need(intervals, k=2):
    iv = sorted(intervals, key=lambda x: (x[1], -x[0]))
    n = len(iv)
    need = [k] * n
    chosen = set()
    print("thứ tự:", iv)
    for i, (s, e) in enumerate(iv):
        p = e
        while need[i] > 0:
            while p in chosen:
                p -= 1
            chosen.add(p)
            for j in range(i, n):
                if iv[j][0] <= p <= iv[j][1]:
                    need[j] -= 1
            print(f"  đoạn {iv[i]}: chọn {p:>2}, need = {need}")
    print("tập chọn:", sorted(chosen), "kích thước", len(chosen))

trace_need([[1, 3], [3, 7], [8, 9]])

thứ tự: [[1, 3], [3, 7], [8, 9]]
  đoạn [1, 3]: chọn  3, need = [1, 1, 2]
  đoạn [1, 3]: chọn  2, need = [0, 1, 2]
  đoạn [3, 7]: chọn  7, need = [0, 0, 2]
  đoạn [8, 9]: chọn  9, need = [0, 0, 1]
  đoạn [8, 9]: chọn  8, need = [0, 0, 0]
tập chọn: [2, 3, 7, 8, 9] kích thước 5


## 9. Code nộp LeetCode

Bản chọn: **Cách 2** — ngắn, `O(n log n)`.

In [17]:
class Solution:
    def intersectionSizeTwo(self, intervals: List[List[int]]) -> int:
        # end tăng dần; cùng end thì đoạn ngắn hơn (start lớn hơn) trước
        intervals.sort(key=lambda x: (x[1], -x[0]))
        a = b = -1          # hai điểm lớn nhất đã chọn, a < b
        ans = 0
        for s, e in intervals:
            if s <= a:          # đoạn đã chứa cả a và b
                continue
            if s <= b:          # chỉ chứa b -> thêm e
                ans += 1
                a, b = b, e
            else:               # không chứa điểm nào -> thêm e-1, e
                ans += 2
                a, b = e - 1, e
        return ans

for iv, exp in EXAMPLES:
    assert Solution().intersectionSizeTwo([r[:] for r in iv]) == exp
print("OK")

OK


Giải thích từng dòng:

- `intervals.sort(...)`: sắp tại chỗ (LeetCode cho phép sửa input). Khoá `(end, -start)`.
- `a = b = -1`: chưa chọn điểm nào; `-1` nhỏ hơn mọi `start >= 0` nên đoạn đầu rơi vào trường hợp "thêm 2".
- `if s <= a`: vì `a < b <= e`, đoạn chứa cả hai.
- `a, b = b, e`: điểm lớn nhất cũ trở thành điểm lớn thứ hai; `e > b` chắc chắn (xem mục 10), nên không trùng.
- `a, b = e - 1, e`: `e - 1 >= s` vì `s < e`.

## 10. Vì sao thuật toán đúng

**Bổ đề 1 (không trùng điểm).** Ở trường hợp `a < s <= b`, ta có `b < e`. Thật vậy, nếu `b = e` thì `b` được thêm khi xử lý một đoạn trước đó `[s', e]` cùng `end`; do sắp `start` giảm dần, `s' >= s`. Sau đoạn đó, `a >= s'` (nếu thêm 2 điểm thì `a = e - 1 >= s'`; nếu thêm 1 thì `a` = `b` cũ `>= s'`). Suy ra `a >= s`, mâu thuẫn với `s > a`.

**Bổ đề 2 (lựa chọn tham lam an toàn — lập luận trao đổi).** Xét một lời giải tối ưu `OPT` phủ các đoạn đã xử lý và đoạn hiện tại. Giả sử đoạn hiện tại `[s, e]` thiếu `r` điểm sau khi xét các điểm tham lam đã chọn. `OPT` phải có ít nhất `r` điểm trong `[s, e]` ngoài các điểm "tương ứng". Thay mỗi điểm như vậy bằng một điểm lớn hơn trong `{e - 1, e}` (chưa có): mọi đoạn **trước** vẫn được phủ (các điểm tham lam cũ đã đủ cho chúng), và mọi đoạn **sau** có `end >= e` nên điểm lớn hơn chỉ có thể nằm trong **nhiều hơn** các đoạn sau. Không tăng kích thước → tham lam không thua.

**Bổ đề 3 (chỉ cần `a, b`).** Mọi điểm đã chọn `<= e`. Đoạn `[s, e]` chứa đúng các điểm đã chọn `>= s`. Vì các điểm đã chọn là `... < a < b`, đoạn chứa ≥ 2 điểm ⇔ `s <= a`; chứa đúng 1 ⇔ `a < s <= b`.

Kết hợp: thuật toán dùng số điểm ít nhất. ∎

**Cách 4** đúng theo lý thuyết ràng buộc hiệu: tập các `P` thoả mãn là tập lời giải của hệ bất đẳng thức; giá trị nhỏ nhất của `P(x)` khi `P(c_0) = 0` là đường đi dài nhất (không có chu trình dương vì hệ khả thi).

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^L · n · L)` (L = độ rộng toạ độ) | `O(L)` |
| 2. Tham lam (a, b) | `O(n log n)` | `O(n)` cho sắp xếp |
| 3. Tham lam `need[]` | `O(n^2)` (+ `O(n log n)`) | `O(n)` |
| 4. Ràng buộc hiệu | `O(n^2)` xấu nhất với quét xuôi/ngược; Bellman-Ford thuần `O(n · E)` | `O(n)` |

Với `n = 3000`: Cách 2 chỉ vài mili giây.

In [18]:
import random, time
random.seed(757)
big = []
for _ in range(3000):
    s = random.randint(0, 10 ** 8 - 1)
    big.append([s, random.randint(s + 1, min(10 ** 8, s + 10 ** 6))])

for name, f in [("greedy_two", greedy_two), ("greedy_need", greedy_need), ("diff", diff_constraints)]:   # diff chậm nhất
    t = time.perf_counter()
    r = f([x[:] for x in big])
    print(f"{name:12} ans={r}  {1000 * (time.perf_counter() - t):8.1f} ms")

greedy_two   ans=906       1.6 ms


greedy_need  ans=906     110.0 ms


diff         ans=906     870.8 ms


## 12. Bộ test

In [19]:
# 12.1 Ví dụ đề cho mọi cách
methods = {
    "brute": brute,
    "greedy_two": greedy_two,
    "greedy_need": greedy_need,
    "diff": diff_constraints,
    "SolutionGreedy": lambda iv: SolutionGreedy().intersectionSizeTwo([r[:] for r in iv]),
    "SolutionNeed": lambda iv: SolutionNeed().intersectionSizeTwo([r[:] for r in iv]),
    "SolutionDiff": lambda iv: SolutionDiff().intersectionSizeTwo([r[:] for r in iv]),
    "Solution": lambda iv: Solution().intersectionSizeTwo([r[:] for r in iv]),
}
for name, f in methods.items():
    for iv, exp in EXAMPLES:
        assert f(iv) == exp, (name, iv)

# 12.2 Edge cases
edge = [
    ([[0, 1]], 2),                          # một đoạn, đúng 2 số
    ([[0, 100000000]], 2),                  # đoạn rất rộng
    ([[1, 2], [1, 2], [1, 2]], 2),          # trùng nhau
    ([[1, 2], [3, 4]], 4),                  # rời nhau
    ([[1, 2], [2, 3]], 3),                  # chung một điểm
    ([[1, 5], [2, 5], [3, 5], [4, 5]], 2),  # cùng end -> chọn 4, 5
    ([[1, 10], [2, 3], [5, 6], [8, 9]], 6),
]
for name, f in methods.items():
    for iv, exp in edge:
        if name == "brute" and max(e for _, e in iv) > 20:
            continue
        assert f(iv) == exp, (name, iv, f(iv))
print("Ví dụ + edge cases: OK")

Ví dụ + edge cases: OK


In [20]:
# 12.3 Đối chiếu ngẫu nhiên với brute force (toạ độ nhỏ)
random.seed(2024)
def rand_case(max_n=6, max_x=11):
    n = random.randint(1, max_n)
    res = []
    for _ in range(n):
        s = random.randint(0, max_x - 1)
        e = random.randint(s + 1, max_x)
        res.append([s, e])
    return res

for _ in range(400):
    iv = rand_case()
    b = brute(iv)
    for name, f in methods.items():
        if name == "brute":
            continue
        assert f(iv) == b, (name, iv, f(iv), b)

# Kiểm tra tập điểm của Cách 3 thật sự phủ
for _ in range(400):
    iv = rand_case(10, 30)
    cnt, pts = greedy_need(iv, return_points=True)
    assert check_cover(iv, pts) and cnt == len(pts) == greedy_two(iv)
print("Đối chiếu ngẫu nhiên: OK")

Đối chiếu ngẫu nhiên: OK


## 13. Lỗi người mới hay gặp

Ta viết một hàm tìm phản ví dụ nhỏ nhất bằng cách so sánh với brute force — kỹ năng rất hữu ích khi gỡ lỗi tham lam.

In [21]:
def find_counterexample(f, trials=3000, seed=1):
    rnd = random.Random(seed)
    best = None
    for _ in range(trials):
        n = rnd.randint(1, 4)
        iv = []
        for _ in range(n):
            s = rnd.randint(0, 6)
            iv.append([s, rnd.randint(s + 1, 7)])
        if f([r[:] for r in iv]) != brute(iv):
            if best is None or len(iv) < len(best):
                best = iv
    return best

### Lỗi 1 — Hoà `end` nhưng sắp `start` **tăng dần**

**Code sai:**

In [22]:
def wrong_tie(intervals):
    a = b = -1
    ans = 0
    for s, e in sorted(intervals, key=lambda x: (x[1], x[0])):   # SAI: start tăng
        if s <= a:
            continue
        if s <= b:
            ans += 1; a, b = b, e
        else:
            ans += 2; a, b = e - 1, e
    return ans

ce = find_counterexample(wrong_tie)
print("phản ví dụ:", ce, "wrong =", wrong_tie(ce), "đúng =", brute(ce))

phản ví dụ: [[5, 6], [3, 5], [0, 2], [2, 5]] wrong = 4 đúng = 5


**Vì sao sai:** xem phản ví dụ: sau `[0,2]` có `a, b = 1, 2`. Đoạn dài `[2,5]` được xử lý trước, thêm `5` → `a, b = 2, 5`. Đoạn ngắn `[3,5]` có `s = 3 > a` nên lại "thêm `5`" — điểm **trùng**! Giờ `a = b = 5`, và đoạn `[5,6]` thấy `s <= a` nên tưởng đã đủ 2 điểm, trong khi thực ra chỉ có điểm `5` → **đếm thiếu** (4 thay vì 5).

**Code đúng:** khoá `(end, -start)` như Cách 2.

### Lỗi 2 — Thêm điểm **nhỏ nhất** (`s`, `s+1`) thay vì lớn nhất

**Code sai:**

In [23]:
def wrong_left_points(intervals):
    chosen = []
    for s, e in sorted(intervals, key=lambda x: (x[1], -x[0])):
        have = sum(1 for x in chosen if s <= x <= e)
        p = s
        while have < 2:
            if p not in chosen:
                chosen.append(p); have += 1
            p += 1                                          # SAI: lấy từ trái
    return len(chosen)

ce = find_counterexample(wrong_left_points)
print("phản ví dụ:", ce, "wrong =", wrong_left_points(ce), "đúng =", brute(ce))

phản ví dụ: [[3, 6], [6, 7]] wrong = 4 đúng = 3


**Vì sao sai:** các đoạn sau có `end` lớn hơn nên điểm ở **bên phải** mới có khả năng dùng chung. Điểm bên trái "lãng phí".

**Code đúng:** lấy `e`, `e - 1`, ... (Cách 2/3).

### Lỗi 3 — Cập nhật sai thứ tự `a, b` khi thêm 1 điểm

**Code sai:**

In [24]:
def wrong_update(intervals):
    a = b = -1
    ans = 0
    for s, e in sorted(intervals, key=lambda x: (x[1], -x[0])):
        if s <= a:
            continue
        if s <= b:
            ans += 1
            b = e                       # SAI: quên đẩy b cũ xuống a
        else:
            ans += 2; a, b = e - 1, e
    return ans

ce = find_counterexample(wrong_update)
print("phản ví dụ:", ce, "wrong =", wrong_update(ce), "đúng =", brute(ce))

phản ví dụ: [[6, 7], [6, 7], [0, 6]] wrong = 4 đúng = 3


**Vì sao sai:** sau khi thêm `e`, hai điểm lớn nhất là `b` (cũ) và `e`. Giữ `a` cũ làm ta tưởng đoạn sau có `s <= a` đã đủ 2 điểm, trong khi thực ra chỉ có 1 → **đếm thiếu**.

**Code đúng:** `a, b = b, e`.

### Lỗi 4 — Chỉ nhớ **một** điểm như bài 452

**Code sai:**

In [25]:
def wrong_one_point(intervals):
    return 2 * min_points_one(intervals)      # SAI: "mỗi đoạn 1 điểm, nhân đôi"

ce = find_counterexample(wrong_one_point)
print("phản ví dụ:", ce, "wrong =", wrong_one_point(ce), "đúng =", brute(ce))

phản ví dụ: [[3, 6], [6, 7]] wrong = 2 đúng = 3


**Vì sao sai:** với `[3,6],[6,7]`, bài 1 điểm chỉ cần điểm `6` (phủ cả hai) → nhân đôi ra 2, nhưng thực tế cần 3 điểm (ví dụ `5, 6, 7`). Ngược lại, có trường hợp nhân đôi lại **thừa**. Quan hệ giữa hai bài không đơn giản là "gấp đôi".

**Code đúng:** theo dõi riêng hai điểm lớn nhất.

## 14. Mẹo & nhận dạng pattern

- **"Chọn ít điểm nhất để mỗi đoạn chứa ≥ k điểm"** → sắp theo `end`, chọn điểm **càng phải càng tốt**.
- **Đoạn + tham lam** → hầu như luôn sắp theo `end` (bài chọn điểm, bài chọn nhiều đoạn rời nhau nhất) hoặc `start` (bài gộp đoạn).
- **Hoà khoá chính** → luôn tự hỏi: thứ tự hoà có ảnh hưởng không? Ở đây có!
- **Chỉ nhớ "biên"**: trạng thái cần thiết thường chỉ là vài phần tử cực trị (`a, b`).
- **Gỡ lỗi tham lam**: viết brute force + tìm phản ví dụ ngẫu nhiên nhỏ nhất.
- **Bất đẳng thức trên tổng tiền tố** → ràng buộc hiệu → đường đi ngắn/dài nhất.

## 15. Biến thể

1. **Mỗi đoạn cần ≥ 1 điểm** (LeetCode 452) — chỉ nhớ 1 điểm.
2. **Mỗi đoạn cần ≥ `k` điểm** (k cố định) — dùng Cách 3, hoặc giữ `k` điểm lớn nhất trong deque.
3. **Mỗi đoạn `i` cần ≥ `c_i` điểm khác nhau** (bài "Intervals" kinh điển POJ 1201) — Cách 3 với `need[i] = c_i`, hoặc ràng buộc hiệu (Cách 4) — cách 4 chính là lời giải chuẩn của bài này.
4. **Chỉ cần giao với nums có kích thước đúng 2** — khác hẳn, không phải tham lam này.
5. **Trả về luôn tập điểm** — Cách 3 với `return_points=True`.

In [26]:
# Biến thể 2 bằng Cách 3 với k = 3, đối chiếu vét cạn nhỏ
def brute_k(intervals, k):
    lo = min(s for s, _ in intervals); hi = max(e for _, e in intervals)
    pts = list(range(lo, hi + 1))
    for r in range(len(pts) + 1):
        for comb in combinations(pts, r):
            if all(sum(1 for x in comb if s <= x <= e) >= k for s, e in intervals):
                return r

random.seed(3)
for _ in range(200):
    iv = []
    for _ in range(random.randint(1, 4)):
        s = random.randint(0, 7)
        iv.append([s, random.randint(s + 2, 10)])        # đảm bảo >= 3 số
    assert greedy_need(iv, k=3) == brute_k(iv, 3), iv
print("Biến thể k = 3: OK")

Biến thể k = 3: OK


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi tập con theo kích thước | `O(2^L·n·L)` | `O(L)` | Chuẩn vàng để test |
| 2. Tham lam (a, b) | Sắp `(end, -start)`, nhớ 2 điểm lớn nhất | `O(n log n)` | `O(n)` | Nộp bài, phỏng vấn |
| 3. Tham lam `need[]` | Chọn điểm từ `end` xuống, giảm `need` | `O(n^2)` | `O(n)` | Khi cần `k` tổng quát / in tập điểm |
| 4. Ràng buộc hiệu | `P(e+1)-P(s) >= 2`, đường đi dài nhất | `O(n^2)`+ | `O(n)` | Khi mỗi đoạn có yêu cầu riêng `c_i` |

## 17. Bài tập

**Bài 1 (đoán output).** `intersectionSizeTwo([[1,3],[4,9],[0,10],[6,7],[1,2],[0,6],[7,9],[0,1],[2,5],[6,8]])` = ?

**Bài 2 (tính tay).** Với `[[0,4],[1,2],[3,5],[2,3]]`, viết thứ tự sau khi sắp và bảng `(a, b, ans)` sau mỗi đoạn.

**Bài 3 (điền TODO).** Hoàn thiện `min_points_k(intervals, k)` dùng ý tưởng giữ **danh sách `k` điểm lớn nhất**.

**Bài 4 (tự viết lại).** Viết lại Cách 2 nhưng **trả về tập điểm** được chọn.

**Bài 5 (mini challenge).** Mỗi đoạn `i` có yêu cầu riêng `c_i` (với `1 <= c_i <= e_i - s_i + 1`). Viết hàm tìm số điểm ít nhất, đối chiếu brute force.

In [27]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def min_points_k(intervals, k):
    # TODO: sắp (end, -start); top = [] (k điểm lớn nhất, tăng dần)
    # đếm have = số phần tử của top >= s; thêm (k - have) điểm lớn nhất chưa có trong đoạn
    pass

# Bài 4 — TODO
def greedy_two_points(intervals):
    # TODO: như greedy_two nhưng lưu các điểm vào list
    pass

# Bài 5 — TODO
def min_points_custom(intervals, c):
    # TODO: Cách 3 với need[i] = c[i] (nhớ sắp kèm chỉ số)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: sắp theo `(end, -start)`.
- Mức 2: đoạn đầu `[0,1]` → chọn 0, 1; rồi `[1,2]` → thêm 2; ...
- Mức 3: đáp án `7`.

**Bài 2.**
- Mức 1: thứ tự: `[1,2], [2,3], [0,4], [3,5]`.
- Mức 2: `[1,2]` → (1,2), ans 2; `[2,3]` → (2,3), ans 3; `[0,4]` → bỏ qua.
- Mức 3: `[3,5]`: `a=2 < 3 <= b=3` → thêm 5, ans 4.

**Bài 3.**
- Mức 1: `top` luôn tăng dần, giữ tối đa `k` phần tử cuối.
- Mức 2: `have = sum(x >= s for x in top)`.
- Mức 3: thêm các điểm `e, e-1, ...` (bỏ qua điểm đã có trong `top`), rồi cắt `top = sorted(top)[-k:]`.

**Bài 4.**
- Mức 1: mỗi khi `ans += 1` thì `pts.append(e)`.
- Mức 2: khi `ans += 2` thì thêm `e-1, e`.
- Mức 3: kiểm tra bằng `check_cover`.

**Bài 5.**
- Mức 1: sắp chỉ số theo `(end, -start)`.
- Mức 2: `need[i] = c[i]`, chọn từ `end` xuống, bỏ qua điểm đã chọn.
- Mức 3: giảm `need[j]` cho mọi `j >= i` có `start_j <= p`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
B1 = [[1,3],[4,9],[0,10],[6,7],[1,2],[0,6],[7,9],[0,1],[2,5],[6,8]]
assert Solution().intersectionSizeTwo([r[:] for r in B1]) == 7 == greedy_need(B1)

# Bài 2
assert sorted([[0,4],[1,2],[3,5],[2,3]], key=lambda x: (x[1], -x[0])) == [[1,2],[2,3],[0,4],[3,5]]
assert greedy_two([[0,4],[1,2],[3,5],[2,3]]) == 4

# Bài 3
def min_points_k_ans(intervals, k):
    top, ans = [], 0
    for s, e in sorted(intervals, key=lambda x: (x[1], -x[0])):
        have = sum(1 for x in top if x >= s)
        p = e
        while have < k:
            if p not in top:
                top.append(p); ans += 1; have += 1
            p -= 1
        top = sorted(top)[-k:]
    return ans

random.seed(17)
for _ in range(300):
    iv = rand_case(5, 10)
    assert min_points_k_ans(iv, 2) == brute(iv), iv
    assert min_points_k_ans(iv, 1) == min_points_one(iv)

# Bài 4
def greedy_two_points_ans(intervals):
    a = b = -1
    pts = []
    for s, e in sorted(intervals, key=lambda x: (x[1], -x[0])):
        if s <= a:
            continue
        if s <= b:
            pts.append(e); a, b = b, e
        else:
            pts += [e - 1, e]; a, b = e - 1, e
    return pts

for _ in range(300):
    iv = rand_case(8, 25)
    pts = greedy_two_points_ans(iv)
    assert len(set(pts)) == len(pts) == greedy_two(iv) and check_cover(iv, pts)

# Bài 5
def min_points_custom_ans(intervals, c):
    order = sorted(range(len(intervals)), key=lambda i: (intervals[i][1], -intervals[i][0]))
    need = [c[i] for i in order]
    iv = [intervals[i] for i in order]
    chosen = set()
    for i, (s, e) in enumerate(iv):
        p = e
        while need[i] > 0:
            while p in chosen:
                p -= 1
            chosen.add(p)
            for j in range(i, len(iv)):
                if iv[j][0] <= p:
                    need[j] -= 1
    return len(chosen)

def brute_custom(intervals, c):
    lo = min(s for s, _ in intervals); hi = max(e for _, e in intervals)
    pts = list(range(lo, hi + 1))
    for r in range(len(pts) + 1):
        for comb in combinations(pts, r):
            if all(sum(1 for x in comb if s <= x <= e) >= ci for (s, e), ci in zip(intervals, c)):
                return r

random.seed(5)
for _ in range(300):
    iv = rand_case(4, 9)
    c = [random.randint(1, e - s + 1) for s, e in iv]
    assert min_points_custom_ans(iv, c) == brute_custom(iv, c), (iv, c)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
intervals.sort(key=lambda x: (x[1], -x[0]))   # end tăng, start giảm
a = b = -1; ans = 0                           # 2 điểm lớn nhất
for s, e in intervals:
    if s <= a: continue                       # đủ 2
    if s <= b: ans += 1; a, b = b, e          # thiếu 1 -> thêm e
    else:      ans += 2; a, b = e - 1, e      # thiếu 2 -> thêm e-1, e
return ans
```

**Checklist trước khi submit:**

- [ ] Sắp theo `end` tăng, **hoà thì `start` giảm**.
- [ ] Khởi tạo `a = b = -1` (nhỏ hơn mọi toạ độ ≥ 0).
- [ ] So sánh `s <= a`, `s <= b` dùng **`<=`** (đoạn đóng).
- [ ] Thêm 1 điểm: `a, b = b, e` (đẩy `b` xuống).
- [ ] Thêm 2 điểm: `e - 1` và `e`.
- [ ] Đã thử: đoạn trùng nhau, cùng `end`, chỉ chung 1 điểm, rời nhau.
- [ ] Tên method đúng: `intersectionSizeTwo(self, intervals)`.